# Generación, limpieza y transformación de un DataSet el principio GIGO

Fecha: 28 de mayo del 2026

Autor: Maria Reina Zarate Nava

# Objetivo general
Identificar problemas de calidad de datos, aplicar técnicas de limpieza y transformación,
y analizar cómo los datos incorrectos afectan el análisis y los modelos de Machine Learning mediante el principio GIGO.

## 1. Importación de Librerías y Carga de Datos
A continuación, cargamos los datos desde la carpeta `DataSets`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
file_path = '../DataSets/ventas-por-factura.csv'
df = pd.read_csv(file_path)
df.head()

## 2. Identificación de Anomalías y Problemas de Calidad

In [ ]:
print("=== Valores Nulos ===")
print(df.isnull().sum())

print("\n=== Tipos de Datos ===")
print(df.dtypes)

print("\n=== Duplicados ===")
print(f"Filas duplicadas: {df.duplicated().sum()}")

### Gráfica 2: Heatmap de Valores Nulos

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Heatmap de Valores Nulos')
plt.show()

### Problemas de Formato y Transformación Inicial
El campo 'Monto' usa comas como separador decimal. Debemos transformarlo a tipo float.

In [ ]:
# Rename columns to english for standard coding (User Rules)
df = df.rename(columns={
    'N° de factura': 'invoice_no',
    'Fecha de factura': 'invoice_date',
    'ID Cliente': 'client_id',
    'País': 'country',
    'Cantidad': 'quantity',
    'Monto': 'amount'
})

# Format 'amount' to float
df['amount'] = df['amount'].str.replace(',', '.').astype(float)

# Format 'invoice_date' to datetime
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

df.head()

### Identificación de Facturas Canceladas y Outliers
Las facturas canceladas inician con 'C'. Cantidades o montos negativos también se consideran anomalías o devoluciones.

In [ ]:
# Cancelled invoices
cancelled_invoices = df[df['invoice_no'].str.startswith('C', na=False)]
print(f"Facturas canceladas: {len(cancelled_invoices)}")

# Negative or zero quantity/amount (Outliers/Invalid)
invalid_quantities = df[df['quantity'] <= 0]
invalid_amounts = df[df['amount'] <= 0]
print(f"Cantidades inválidas (<=0): {len(invalid_quantities)}")
print(f"Montos inválidos (<=0): {len(invalid_amounts)}")

### Gráfica 3: Distribución del Monto (Antes de Limpiar)

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['amount'])
plt.title('Distribución del Monto (Con Outliers)')
plt.show()

## 3. Limpieza de Datos
Aplicaremos las siguientes reglas:
1. Eliminar filas con valores nulos (o imputar, en este caso eliminaremos registros sin ID de cliente).
2. Eliminar facturas canceladas.
3. Filtrar cantidades y montos negativos o iguales a cero.
4. Eliminar duplicados.

In [ ]:
# Remove duplicates
df_clean = df.drop_duplicates()

# Drop null values in critical columns
df_clean = df_clean.dropna(subset=['client_id'])

# Remove cancelled invoices
df_clean = df_clean[~df_clean['invoice_no'].str.startswith('C', na=False)]

# Filter out valid quantities and amounts
df_clean = df_clean[(df_clean['quantity'] > 0) & (df_clean['amount'] > 0)]

print(f"Filas antes de limpieza: {len(df)}")
print(f"Filas después de limpieza: {len(df_clean)}")

### Gráfica 1: Ventas por País (Top 10)
Usando el dataset limpio.

In [ ]:
sales_by_country = df_clean.groupby('country')['amount'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=sales_by_country.values, y=sales_by_country.index, palette='Blues_r')
plt.title('Top 10 Ventas por País')
plt.xlabel('Monto Total')
plt.ylabel('País')
plt.show()

### Gráfica 3: Distribución del Monto (Después de Limpiar)

In [ ]:
# Filter extreme outliers for better visualization
q1 = df_clean['amount'].quantile(0.25)
q3 = df_clean['amount'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
df_filtered = df_clean[df_clean['amount'] <= upper_bound]

plt.figure(figsize=(10, 6))
sns.histplot(df_filtered['amount'], bins=50, kde=True)
plt.title('Distribución del Monto (Sin Outliers Extremos)')
plt.xlabel('Monto')
plt.ylabel('Frecuencia')
plt.show()

## 4. Exportar Dataset Limpio
Guardamos el resultado en la carpeta `DataSet`.

In [ ]:
output_path = '../DataSet/Data_Limpio_Factura.csv'
df_clean.to_csv(output_path, index=False)
print("Datos guardados exitosamente.")

## 5. Justificación de Decisiones y Principio GIGO

### Justificación de Decisiones:
- **Valores Nulos**: Se eliminaron los registros sin `ID Cliente` porque no aportan valor para un análisis centrado en el comportamiento de clientes o ventas rastreables.
- **Facturas Canceladas y Valores Negativos**: Se eliminaron debido a que representan devoluciones o errores, y ensucian los modelos predictivos de ventas netas.
- **Transformación de Tipos**: El `Monto` como string con coma decimal impedía operaciones matemáticas. Se transformó a float para permitir agrupaciones y gráficas. La fecha se transformó a `datetime` para eventuales análisis temporales.

### Principio GIGO (Garbage In, Garbage Out):
El principio GIGO establece que la calidad del resultado de un modelo de Machine Learning o análisis estadístico depende enteramente de la calidad de los datos de entrada. 

En este laboratorio se observó cómo los montos con formato incorrecto, datos nulos o facturas negativas (basura) distorsionan métricas clave (como ventas totales). Si estos datos se ingresaran a un modelo de predicción de ingresos, el modelo aprendería patrones erróneos y generaría predicciones inútiles. Limpiar y transformar el dataset garantiza conclusiones válidas.